In [1]:
"""
Single vs Multi-Species Modelling — COMBINED SCRIPT (all features + GAM)
EcoStat Modelling — DATA5925

This merges everything built across the separate scripts into one file:
  - single_vs_multi_species.py  (baseline: single / multi / multi+interactions)
  - 01_repeated_spatial_cv.py   (repeated random CV + spatial block CV)
  - 02_hyperparameter_tuning.py (fair tuning per approach)
  - 03_feature_importance.py   (permutation importance, why pooling helps/hurts)
  - 04_full_metrics_heatmap.py (Brier/F1/sensitivity/specificity + heatmaps)

...and adds GAM (via pygam) as a fifth model alongside LogisticRegression,
RandomForest, MLP, and XGBoost — matching Ruiye's assignment in the Project
Plan. Install with: pip install pygam

GAM is wrapped in a small sklearn-compatible class (PygamLogisticGAM) so it
drops into the exact same Pipelines/ColumnTransformers/RandomizedSearchCV
calls as every other model. It fits a smooth spline term per environmental
predictor, and (for the pooled multi-species model) a linear term per
one-hot Species column.

WHAT RUNS
---------
Toggle these to control runtime — running everything on 8 species x 5
models x several CV repeats is a lot of model fits.

    RUN_BASELINE          -> single vs multi vs multi+interactions (fast)
    RUN_REPEATED_SPATIAL_CV -> robust estimates via resampling (slow)
    RUN_HYPERPARAM_TUNING -> fair tuned comparison (slow)
    RUN_FEATURE_IMPORTANCE -> why pooling helps/hurts (medium)
    RUN_FULL_METRICS      -> Brier/F1/sensitivity/specificity + heatmaps (fast)

Everything reads from TRAIN_PATH / TEST_PATH and writes to OUTPUT_DIR below.
"""


"\nSingle vs Multi-Species Modelling — COMBINED SCRIPT (all features + GAM)\nEcoStat Modelling — DATA5925\n\nThis merges everything built across the separate scripts into one file:\n  - single_vs_multi_species.py  (baseline: single / multi / multi+interactions)\n  - 01_repeated_spatial_cv.py   (repeated random CV + spatial block CV)\n  - 02_hyperparameter_tuning.py (fair tuning per approach)\n  - 03_feature_importance.py   (permutation importance, why pooling helps/hurts)\n  - 04_full_metrics_heatmap.py (Brier/F1/sensitivity/specificity + heatmaps)\n\n...and adds GAM (via pygam) as a fifth model alongside LogisticRegression,\nRandomForest, MLP, and XGBoost — matching Ruiye's assignment in the Project\nPlan. Install with: pip install pygam\n\nGAM is wrapped in a small sklearn-compatible class (PygamLogisticGAM) so it\ndrops into the exact same Pipelines/ColumnTransformers/RandomizedSearchCV\ncalls as every other model. It fits a smooth spline term per environmental\npredictor, and (for 

In [2]:

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    log_loss, roc_auc_score, brier_score_loss, f1_score,
    confusion_matrix, make_scorer
)
from scipy.stats import spearmanr

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[warning] xgboost not installed — skipping XGBoost. pip install xgboost")

try:
    from pygam import LogisticGAM, s, l
    HAS_GAM = True
except ImportError:
    HAS_GAM = False
    print("[warning] pygam not installed — skipping GAM. pip install pygam")

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/All model/outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"
PLOT_COL = "plot"
NUMERIC_FEATURES = ["easting", "northing", "disturb", "rainann",
                     "soildepth", "soilfert", "tempann", "topo"]
CLIP = 1e-6

RUN_BASELINE = True
RUN_REPEATED_SPATIAL_CV = True
RUN_HYPERPARAM_TUNING = True
RUN_FEATURE_IMPORTANCE = True
RUN_FULL_METRICS = True

# Runtime knobs (reduce these first if things are too slow)
N_SPLITS_RANDOM, N_REPEATS_RANDOM = 5, 3
N_BLOCKS_SPATIAL, N_REPEATS_SPATIAL = 6, 3
N_ITER_SINGLE, N_ITER_MULTI, MIN_POS_TO_TUNE = 15, 30, 6
N_REPEATS_PERM = 20
IMPORTANCE_REFERENCE_MODEL = "RandomForest"  # which model to explain in step 3


# ----------------------------------------------------------------------
# GAM WRAPPER — makes pygam behave like any other sklearn classifier
# ----------------------------------------------------------------------
class PygamLogisticGAM(BaseEstimator, ClassifierMixin):
    """
    Wraps pygam.LogisticGAM as a standard sklearn estimator.

    Assumes columns are ordered [smooth numeric features] followed by
    [one-hot / other columns treated as linear terms] — this matches the
    ColumnTransformer order used everywhere in this script (numeric first,
    categorical after). For the single-species approach there are no
    categorical columns, so n_smooth = total number of columns.
    """
    def __init__(self, n_smooth=8, lam=0.6, n_splines=10):
        self.n_smooth = n_smooth
        self.lam = lam
        self.n_splines = n_splines

    def fit(self, X, y):
        X = np.asarray(X)
        n_features = X.shape[1]
        n_smooth = min(self.n_smooth, n_features)
        terms = None
        for i in range(n_features):
            term = s(i, n_splines=self.n_splines) if i < n_smooth else l(i)
            terms = term if terms is None else terms + term

        # pygam's PIRLS optimizer can diverge for some (lam, n_splines)
        # combinations, especially on small/imbalanced per-species subsets
        # during hyperparameter search. Rather than let this crash the whole
        # script (it happens inside RandomizedSearchCV's final refit, which
        # isn't covered by error_score=), retry with progressively stronger
        # regularization, and fall back to a safe default as a last resort.
        from pygam.utils import OptimizationError
        for attempt_lam in (self.lam, max(self.lam * 10, 1.0), 10.0, 100.0):
            try:
                self.model_ = LogisticGAM(terms, lam=attempt_lam)
                self.model_.fit(X, np.asarray(y))
                self.classes_ = np.array([0, 1])
                return self
            except OptimizationError:
                continue
        # last resort: intercept-only-ish fit with heavy smoothing, should
        # always converge
        self.model_ = LogisticGAM(terms, lam=1000.0)
        self.model_.fit(X, np.asarray(y))
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p1 = np.clip(self.model_.predict_proba(np.asarray(X)), CLIP, 1 - CLIP)
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ----------------------------------------------------------------------
# MODEL ZOO
# ----------------------------------------------------------------------
def build_models():
    models = {
        "LogisticRegression": lambda: LogisticRegression(max_iter=2000, class_weight="balanced"),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, class_weight="balanced",
            random_state=42, n_jobs=-1),
        "MLP": lambda: MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3,
                                      max_iter=1500, random_state=42),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
    if HAS_GAM:
        # n_smooth is set correctly at fit-time inside the pipelines below
        # (single-species: all numeric; multi-species: numeric only, since
        # GAM pipelines set n_smooth=len(NUMERIC_FEATURES) explicitly)
        models["GAM"] = lambda: PygamLogisticGAM(n_smooth=len(NUMERIC_FEATURES))
    return models


def model_names():
    return list(build_models().keys())


def maybe_set_xgb_weight(model, y_train):
    if HAS_XGB and isinstance(model, XGBClassifier):
        pos, neg = max(y_train.sum(), 1), max(len(y_train) - y_train.sum(), 1)
        model.set_params(scale_pos_weight=neg / pos)


def make_multi_preprocessor():
    # sparse_output=False is required so GAM (and the interaction-features
    # variant) get a dense array rather than a scipy sparse matrix.
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), [SPECIES_COL]),
    ])


def clipped_log_loss(y_true, y_pred):
    p = np.clip(y_pred, CLIP, 1 - CLIP)
    return log_loss(y_true, p, labels=[0, 1])


def safe_auc(y_true, y_pred):
    return roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan


# ========================================================================
# SECTION 1 — BASELINE: single vs multi vs multi+interactions
# ========================================================================
def run_single_species(train, test, model_fn):
    rows, preds_all = [], np.zeros(len(test))
    for species, train_sp in train.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        test_sp = test.loc[mask]
        if test_sp.empty:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)

        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_train, y_train)
        p = pipe.predict_proba(X_test)[:, 1]
        preds_all[mask] = p

        rows.append({
            "species": species, "n_train": len(train_sp), "n_test": len(test_sp),
            "prevalence_train": y_train.mean(),
            "log_loss_single": clipped_log_loss(y_test, p),
            "auc_single": safe_auc(y_test, p),
        })
    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    overall = clipped_log_loss(test[TARGET].astype(int), preds_all)
    return df, preds_all, overall


def run_multi_species(train, test, model_fn, interactions=False):
    y_train, y_test = train[TARGET].astype(int), test[TARGET].astype(int)

    if interactions:
        species_dummies_train = pd.get_dummies(train[SPECIES_COL], prefix="sp")
        species_dummies_test = pd.get_dummies(test[SPECIES_COL], prefix="sp").reindex(
            columns=species_dummies_train.columns, fill_value=0)
        cross_train, cross_test = pd.DataFrame(index=train.index), pd.DataFrame(index=test.index)
        for col in NUMERIC_FEATURES:
            for sp_col in species_dummies_train.columns:
                cross_train[f"{col}_x_{sp_col}"] = train[col].values * species_dummies_train[sp_col].values
                cross_test[f"{col}_x_{sp_col}"] = test[col].values * species_dummies_test[sp_col].values
        X_train_full = pd.concat([train[NUMERIC_FEATURES].reset_index(drop=True),
                                   species_dummies_train.reset_index(drop=True),
                                   cross_train.reset_index(drop=True)], axis=1)
        X_test_full = pd.concat([test[NUMERIC_FEATURES].reset_index(drop=True),
                                  species_dummies_test.reset_index(drop=True),
                                  cross_test.reset_index(drop=True)], axis=1)
        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_train_full, y_train)
        p = pipe.predict_proba(X_test_full)[:, 1]
    else:
        X_train_raw = train[NUMERIC_FEATURES + [SPECIES_COL]]
        X_test_raw = test[NUMERIC_FEATURES + [SPECIES_COL]]
        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("prep", make_multi_preprocessor()), ("clf", model)])
        pipe.fit(X_train_raw, y_train)
        p = pipe.predict_proba(X_test_raw)[:, 1]

    overall = clipped_log_loss(y_test, p)
    suffix = "multi_int" if interactions else "multi"
    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        rows.append({"species": species,
                     f"log_loss_{suffix}": clipped_log_loss(test_sp[TARGET].astype(int), p[mask]),
                     f"auc_{suffix}": safe_auc(test_sp[TARGET].astype(int), p[mask])})
    return pd.DataFrame(rows).sort_values("species").reset_index(drop=True), p, overall


def plot_model_comparison(merged, model_name, output_dir):
    species = merged["species"]
    x = np.arange(len(species))
    width = 0.25
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width, merged["log_loss_single"], width, label="Single-species")
    ax.bar(x, merged["log_loss_multi"], width, label="Multi-species")
    if "log_loss_multi_int" in merged.columns:
        ax.bar(x + width, merged["log_loss_multi_int"], width, label="Multi + interactions")
    ax.set_xticks(x)
    ax.set_xticklabels(species, rotation=40, ha="right")
    ax.set_ylabel("Log-loss (lower is better)")
    ax.set_title(f"{model_name}: per-species log-loss by approach")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(output_dir, f"{model_name}_comparison.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def section_baseline(train, test, output_dir):
    print("\n" + "=" * 70)
    print("SECTION 1 — BASELINE: single vs multi vs multi+interactions")
    print("=" * 70)
    models = build_models()
    summary_rows = []
    for model_name, model_fn in models.items():
        print(f"Running {model_name} ...")
        single_df, _, single_ll = run_single_species(train, test, model_fn)
        multi_df, _, multi_ll = run_multi_species(train, test, model_fn, interactions=False)
        multi_int_df, _, multi_int_ll = run_multi_species(train, test, model_fn, interactions=True)

        merged = single_df.merge(multi_df, on="species").merge(multi_int_df, on="species")
        merged["best_approach"] = merged[
            ["log_loss_single", "log_loss_multi", "log_loss_multi_int"]
        ].idxmin(axis=1).str.replace("log_loss_", "", regex=False)

        merged.to_csv(os.path.join(output_dir, f"{model_name}_per_species.csv"), index=False)
        plot_model_comparison(merged, model_name, output_dir)

        print(f"  single: {single_ll:.4f} | multi: {multi_ll:.4f} | multi+int: {multi_int_ll:.4f}")
        summary_rows.append({
            "model": model_name, "overall_log_loss_single": single_ll,
            "overall_log_loss_multi": multi_ll, "overall_log_loss_multi_interactions": multi_int_ll,
            "best_overall": min([("single", single_ll), ("multi", multi_ll), ("multi_int", multi_int_ll)],
                                 key=lambda t: t[1])[0],
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(output_dir, "summary_across_models.csv"), index=False)
    print("\n----- Baseline summary -----")
    print(summary_df.to_string(index=False))
    return summary_df


# ========================================================================
# SECTION 2 — REPEATED & SPATIAL CV
# ========================================================================
def fit_predict_single_cv(train_fold, eval_fold, model_fn):
    preds = np.full(len(eval_fold), np.nan)
    for species, tr_sp in train_fold.groupby(SPECIES_COL):
        mask = (eval_fold[SPECIES_COL] == species).values
        ev_sp = eval_fold.loc[mask]
        if ev_sp.empty or tr_sp[TARGET].nunique() < 2:
            continue
        X_tr, y_tr = tr_sp[NUMERIC_FEATURES], tr_sp[TARGET].astype(int)
        model = model_fn()
        maybe_set_xgb_weight(model, y_tr)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_tr, y_tr)
        preds[mask] = pipe.predict_proba(ev_sp[NUMERIC_FEATURES])[:, 1]
    return preds


def fit_predict_multi_cv(train_fold, eval_fold, model_fn):
    X_tr, y_tr = train_fold[NUMERIC_FEATURES + [SPECIES_COL]], train_fold[TARGET].astype(int)
    X_ev = eval_fold[NUMERIC_FEATURES + [SPECIES_COL]]
    model = model_fn()
    maybe_set_xgb_weight(model, y_tr)
    pipe = Pipeline([("prep", make_multi_preprocessor()), ("clf", model)])
    pipe.fit(X_tr, y_tr)
    return pipe.predict_proba(X_ev)[:, 1]


def evaluate_fold(train_fold, eval_fold, model_fn):
    single_preds = fit_predict_single_cv(train_fold, eval_fold, model_fn)
    multi_preds = fit_predict_multi_cv(train_fold, eval_fold, model_fn)
    y_true = eval_fold[TARGET].astype(int).values
    valid = ~np.isnan(single_preds)

    rows = [{"species": "__overall__",
             "log_loss_single": clipped_log_loss(y_true[valid], single_preds[valid]),
             "log_loss_multi": clipped_log_loss(y_true, multi_preds)}]
    for species in eval_fold[SPECIES_COL].unique():
        sp_mask = (eval_fold[SPECIES_COL] == species).values
        single_sp = single_preds[sp_mask]
        if np.all(np.isnan(single_sp)):
            continue
        keep = ~np.isnan(single_sp)
        y_sp = eval_fold.loc[sp_mask, TARGET].astype(int).values
        rows.append({"species": species,
                     "log_loss_single": clipped_log_loss(y_sp[keep], single_sp[keep]),
                     "log_loss_multi": clipped_log_loss(y_sp, multi_preds[sp_mask])})
    return pd.DataFrame(rows)


def random_group_cv_folds(data, n_splits, n_repeats, seed0=0):
    plots = data[PLOT_COL].unique()
    for repeat in range(n_repeats):
        rng = np.random.RandomState(seed0 + repeat)
        shuffled = rng.permutation(plots)
        fold_assign = {p: i % n_splits for i, p in enumerate(shuffled)}
        plot_fold = data[PLOT_COL].map(fold_assign)
        for fold in range(n_splits):
            eval_mask = (plot_fold == fold).values
            yield repeat, fold, data.loc[~eval_mask].copy(), data.loc[eval_mask].copy()


def spatial_block_cv_folds(data, n_blocks, n_repeats, seed0=100):
    plot_coords = data.groupby(PLOT_COL)[["easting", "northing"]].mean()
    for repeat in range(n_repeats):
        km = KMeans(n_clusters=n_blocks, random_state=seed0 + repeat, n_init=10)
        block_labels = km.fit_predict(plot_coords[["easting", "northing"]])
        plot_to_block = dict(zip(plot_coords.index, block_labels))
        block_of_row = data[PLOT_COL].map(plot_to_block)
        for block in range(n_blocks):
            eval_mask = (block_of_row == block).values
            if eval_mask.sum() == 0 or (~eval_mask).sum() == 0:
                continue
            yield repeat, block, data.loc[~eval_mask].copy(), data.loc[eval_mask].copy()


def run_cv_scheme(scheme_name, fold_generator, models):
    folds = list(fold_generator)  # materialise once so every model sees the same folds
    all_rows = []
    for model_name, model_fn in models.items():
        print(f"  [{scheme_name}] {model_name} ...")
        for repeat, fold, train_fold, eval_fold in folds:
            fold_df = evaluate_fold(train_fold, eval_fold, model_fn)
            fold_df["model"], fold_df["repeat"], fold_df["fold"] = model_name, repeat, fold
            all_rows.append(fold_df)
    return pd.concat(all_rows, ignore_index=True)


def summarise_cv(results_df, scheme_name, output_dir):
    summary = results_df.groupby(["model", "species"])[["log_loss_single", "log_loss_multi"]].agg(["mean", "std"])
    summary.columns = ["_".join(c) for c in summary.columns]
    summary = summary.reset_index()
    summary["diff_mean (single - multi)"] = summary["log_loss_single_mean"] - summary["log_loss_multi_mean"]
    summary.to_csv(os.path.join(output_dir, f"{scheme_name}_summary.csv"), index=False)

    overall = summary[summary["species"] == "__overall__"]
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(overall))
    width = 0.35
    ax.bar(x - width / 2, overall["log_loss_single_mean"], width,
           yerr=overall["log_loss_single_std"], capsize=4, label="Single-species")
    ax.bar(x + width / 2, overall["log_loss_multi_mean"], width,
           yerr=overall["log_loss_multi_std"], capsize=4, label="Multi-species")
    ax.set_xticks(x)
    ax.set_xticklabels(overall["model"])
    ax.set_ylabel("Mean log-loss across folds (± std)")
    ax.set_title(f"{scheme_name.replace('_', ' ').title()}: overall log-loss with fold variability")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, f"{scheme_name}_overall.png"), dpi=150)
    plt.close(fig)
    return summary


def section_repeated_spatial_cv(train, test, output_dir):
    print("\n" + "=" * 70)
    print("SECTION 2 — REPEATED & SPATIAL CROSS-VALIDATION")
    print("=" * 70)
    data = pd.concat([train, test], ignore_index=True)
    models = build_models()

    print("Random (grouped-by-plot) CV ...")
    random_results = run_cv_scheme("random_cv", random_group_cv_folds(
        data, N_SPLITS_RANDOM, N_REPEATS_RANDOM), models)
    random_summary = summarise_cv(random_results, "random_cv", output_dir)

    print("Spatial block CV ...")
    spatial_results = run_cv_scheme("spatial_cv", spatial_block_cv_folds(
        data, N_BLOCKS_SPATIAL, N_REPEATS_SPATIAL), models)
    spatial_summary = summarise_cv(spatial_results, "spatial_cv", output_dir)

    print("\n----- Random CV overall (mean ± std) -----")
    print(random_summary[random_summary.species == "__overall__"]
          [["model", "log_loss_single_mean", "log_loss_single_std",
            "log_loss_multi_mean", "log_loss_multi_std"]].to_string(index=False))
    print("\n----- Spatial CV overall (mean ± std) -----")
    print(spatial_summary[spatial_summary.species == "__overall__"]
          [["model", "log_loss_single_mean", "log_loss_single_std",
            "log_loss_multi_mean", "log_loss_multi_std"]].to_string(index=False))


# ========================================================================
# SECTION 3 — HYPERPARAMETER TUNING (fair single vs multi)
# ========================================================================
def clipped_log_loss_scorer():
    def _score(y_true, y_pred_proba):
        return -log_loss(y_true, np.clip(y_pred_proba, CLIP, 1 - CLIP), labels=[0, 1])
    return make_scorer(_score, needs_proba=True, greater_is_better=True)


NEG_LOG_LOSS = clipped_log_loss_scorer()


def get_search_space(model_name):
    if model_name == "LogisticRegression":
        return LogisticRegression(max_iter=3000, class_weight="balanced"), {
            "clf__C": np.logspace(-3, 2, 20), "clf__penalty": ["l2"]}
    if model_name == "RandomForest":
        return RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1), {
            "clf__n_estimators": [200, 400, 600, 800], "clf__max_depth": [None, 4, 6, 8, 12],
            "clf__min_samples_leaf": [1, 2, 3, 5, 10], "clf__max_features": ["sqrt", "log2", None]}
    if model_name == "MLP":
        return MLPClassifier(max_iter=2000, random_state=42), {
            "clf__hidden_layer_sizes": [(16,), (32,), (32, 16), (64, 32)],
            "clf__alpha": np.logspace(-5, -1, 10),
            "clf__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3]}
    if model_name == "XGBoost" and HAS_XGB:
        return XGBClassifier(eval_metric="logloss", random_state=42), {
            "clf__n_estimators": [100, 200, 300, 500], "clf__max_depth": [2, 3, 4, 6],
            "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "clf__subsample": [0.6, 0.8, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0]}
    if model_name == "GAM" and HAS_GAM:
        return PygamLogisticGAM(), {
            "clf__lam": np.logspace(-2, 2, 15), "clf__n_splines": [5, 8, 10, 15, 20]}
    raise ValueError(model_name)


def default_model(model_name):
    if model_name == "LogisticRegression":
        return LogisticRegression(max_iter=3000, class_weight="balanced")
    if model_name == "RandomForest":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                       class_weight="balanced", random_state=42, n_jobs=-1)
    if model_name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(16,), alpha=1e-3, max_iter=1500, random_state=42)
    if model_name == "XGBoost":
        return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                              eval_metric="logloss", random_state=42)
    if model_name == "GAM":
        return PygamLogisticGAM(n_smooth=len(NUMERIC_FEATURES))
    raise ValueError(model_name)


def tune_pipeline(base_estimator, param_dist, X, y, n_iter, cv_splits):
    pipe = Pipeline([("scale", StandardScaler()), ("clf", base_estimator)])
    search = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=n_iter,
                                 scoring=NEG_LOG_LOSS, cv=cv_splits, random_state=42,
                                 n_jobs=-1, error_score=np.nan)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_


def tuned_single_species(train, test, model_name):
    rows, preds_all = [], np.full(len(test), np.nan)
    base_estimator, param_dist = get_search_space(model_name)
    for species, train_sp in train.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        test_sp = test.loc[mask]
        if test_sp.empty:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)
        n_pos = y_train.sum()

        if n_pos < MIN_POS_TO_TUNE:
            model = default_model(model_name)
            pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
            pipe.fit(X_train, y_train)
            best_params = "defaults (too few positives to tune)"
        else:
            n_splits = min(5, n_pos)
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            pipe, best_params = tune_pipeline(base_estimator, param_dist, X_train, y_train,
                                               N_ITER_SINGLE, cv)
        p = pipe.predict_proba(X_test)[:, 1]
        preds_all[mask] = p
        rows.append({"species": species, "n_train": len(train_sp), "n_pos_train": int(n_pos),
                     "log_loss_single_tuned": clipped_log_loss(y_test, p),
                     "best_params_single": str(best_params)})
    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    return df, clipped_log_loss(test[TARGET].astype(int), preds_all)


def tuned_multi_species(train, test, model_name):
    base_estimator, param_dist = get_search_space(model_name)
    X_train_raw = train[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train = train[TARGET].astype(int)
    X_test_raw = test[NUMERIC_FEATURES + [SPECIES_COL]]
    y_test = test[TARGET].astype(int)

    pipe_template = Pipeline([("prep", make_multi_preprocessor()), ("clf", base_estimator)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = RandomizedSearchCV(pipe_template, param_distributions=param_dist, n_iter=N_ITER_MULTI,
                                 scoring=NEG_LOG_LOSS, cv=cv, random_state=42, n_jobs=-1, error_score=np.nan)
    search.fit(X_train_raw, y_train)
    p = search.best_estimator_.predict_proba(X_test_raw)[:, 1]
    overall = clipped_log_loss(y_test, p)

    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        rows.append({"species": species,
                     "log_loss_multi_tuned": clipped_log_loss(test_sp[TARGET].astype(int), p[mask])})
    return pd.DataFrame(rows).sort_values("species").reset_index(drop=True), overall, search.best_params_


def section_hyperparameter_tuning(train, test, output_dir):
    print("\n" + "=" * 70)
    print("SECTION 3 — FAIR HYPERPARAMETER TUNING (single vs multi)")
    print("=" * 70)
    summary_rows = []
    for model_name in model_names():
        print(f"Tuning {model_name} ...")
        single_df, single_ll = tuned_single_species(train, test, model_name)
        multi_df, multi_ll, multi_best_params = tuned_multi_species(train, test, model_name)
        merged = single_df.merge(multi_df, on="species")
        merged.to_csv(os.path.join(output_dir, f"{model_name}_tuned_per_species.csv"), index=False)
        print(f"  tuned single: {single_ll:.4f} | tuned multi: {multi_ll:.4f} "
              f"({'multi wins' if multi_ll < single_ll else 'single wins'})")
        summary_rows.append({"model": model_name, "overall_log_loss_single_tuned": single_ll,
                              "overall_log_loss_multi_tuned": multi_ll,
                              "multi_wins_after_tuning": multi_ll < single_ll})
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(output_dir, "tuned_summary_across_models.csv"), index=False)
    print("\n----- Tuned summary -----")
    print(summary_df.to_string(index=False))


# ========================================================================
# SECTION 4 — FEATURE IMPORTANCE (why pooling helps/hurts)
# ========================================================================
def neg_log_loss_scorer_estimator():
    def _score(estimator, X, y):
        p = np.clip(estimator.predict_proba(X)[:, 1], CLIP, 1 - CLIP)
        return -log_loss(y, p, labels=[0, 1])
    return _score


IMPORTANCE_SCORER = neg_log_loss_scorer_estimator()


def single_species_importance(train, test, model_name):
    model_fn = lambda: default_model(model_name) if model_name != "GAM" else \
        PygamLogisticGAM(n_smooth=len(NUMERIC_FEATURES))
    rows = []
    for species, train_sp in train.groupby(SPECIES_COL):
        test_sp = test.loc[test[SPECIES_COL] == species]
        if test_sp.empty or train_sp[TARGET].nunique() < 2:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model_fn())])
        pipe.fit(X_train, y_train)
        result = permutation_importance(pipe, X_test, y_test, scoring=IMPORTANCE_SCORER,
                                         n_repeats=N_REPEATS_PERM, random_state=42)
        for feat, imp in zip(NUMERIC_FEATURES, result.importances_mean):
            rows.append({"species": species, "feature": feat, "importance": imp})
    return pd.DataFrame(rows)


def multi_species_importance(train, test, model_name):
    model = default_model(model_name) if model_name != "GAM" else \
        PygamLogisticGAM(n_smooth=len(NUMERIC_FEATURES))
    X_train = train[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train = train[TARGET].astype(int)
    pipe = Pipeline([("prep", make_multi_preprocessor()), ("clf", model)])
    pipe.fit(X_train, y_train)

    rows = []
    col_order = NUMERIC_FEATURES + [SPECIES_COL]
    for species, test_sp in test.groupby(SPECIES_COL):
        if test_sp[TARGET].nunique() < 1:
            continue
        X_test_sp = test_sp[col_order]
        y_test_sp = test_sp[TARGET].astype(int)
        result = permutation_importance(pipe, X_test_sp, y_test_sp, scoring=IMPORTANCE_SCORER,
                                         n_repeats=N_REPEATS_PERM, random_state=42)
        for feat, imp in zip(col_order, result.importances_mean):
            if feat == SPECIES_COL:
                continue
            rows.append({"species": species, "feature": feat, "importance": imp})
    return pd.DataFrame(rows)


def plot_importance_heatmap(df, title, path):
    pivot = df.pivot(index="feature", columns="species", values="importance").reindex(NUMERIC_FEATURES)
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=40, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Permutation importance (drop in neg-log-loss)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def compute_agreement(single_df, multi_df):
    rows = []
    for species in single_df["species"].unique():
        s = single_df[single_df["species"] == species].set_index("feature")["importance"]
        m = multi_df[multi_df["species"] == species].set_index("feature")["importance"]
        common = s.index.intersection(m.index)
        if len(common) < 3:
            continue
        rho, _ = spearmanr(s.loc[common], m.loc[common])
        rows.append({"species": species, "single_vs_pooled_importance_agreement": rho})
    return pd.DataFrame(rows)


def section_feature_importance(train, test, output_dir):
    print("\n" + "=" * 70)
    print(f"SECTION 4 — FEATURE IMPORTANCE (reference model: {IMPORTANCE_REFERENCE_MODEL})")
    print("=" * 70)
    single_imp = single_species_importance(train, test, IMPORTANCE_REFERENCE_MODEL)
    single_imp.to_csv(os.path.join(output_dir, "single_species_importance.csv"), index=False)
    multi_imp = multi_species_importance(train, test, IMPORTANCE_REFERENCE_MODEL)
    multi_imp.to_csv(os.path.join(output_dir, "multi_species_importance.csv"), index=False)

    plot_importance_heatmap(single_imp, f"{IMPORTANCE_REFERENCE_MODEL} (single): importance by species",
                             os.path.join(output_dir, "importance_heatmap_single.png"))
    plot_importance_heatmap(multi_imp, f"{IMPORTANCE_REFERENCE_MODEL} (pooled): importance by species",
                             os.path.join(output_dir, "importance_heatmap_multi.png"))

    agreement = compute_agreement(single_imp, multi_imp)
    agreement.to_csv(os.path.join(output_dir, "species_feature_agreement.csv"), index=False)
    print("\n----- Feature-ranking agreement (single vs pooled) -----")
    print(agreement.sort_values("single_vs_pooled_importance_agreement", ascending=False).to_string(index=False))


# ========================================================================
# SECTION 5 — FULL METRICS + HEATMAPS
# ========================================================================
def best_f1_threshold(y_train_true, y_train_pred):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.9, 90):
        calls = (y_train_pred >= t).astype(int)
        f1 = f1_score(y_train_true, calls, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


def compute_binary_metrics(y_true, y_pred_proba, threshold):
    calls = (y_pred_proba >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        return {"f1": np.nan, "sensitivity": np.nan, "specificity": np.nan}
    tn, fp, fn, tp = confusion_matrix(y_true, calls, labels=[0, 1]).ravel()
    return {"f1": f1_score(y_true, calls, zero_division=0),
            "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan}


def full_metrics(y_true, p_test, y_train_true, p_train):
    t_opt = best_f1_threshold(y_train_true, p_train)
    metrics = {"log_loss": clipped_log_loss(y_true, p_test), "auc": safe_auc(y_true, p_test),
               "brier": brier_score_loss(y_true, p_test), "threshold_used": t_opt}
    metrics.update({f"{k}_at_0.5": v for k, v in compute_binary_metrics(y_true, p_test, 0.5).items()})
    metrics.update({f"{k}_at_opt": v for k, v in compute_binary_metrics(y_true, p_test, t_opt).items()})
    return metrics


def run_single_full(train, test, model_fn):
    rows = []
    for species, train_sp in train.groupby(SPECIES_COL):
        test_sp = test.loc[test[SPECIES_COL] == species]
        if test_sp.empty:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)
        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_train, y_train)
        p_test, p_train = pipe.predict_proba(X_test)[:, 1], pipe.predict_proba(X_train)[:, 1]
        m = full_metrics(y_test, p_test, y_train, p_train)
        m["species"], m["approach"] = species, "single"
        rows.append(m)
    return pd.DataFrame(rows)


def run_multi_full(train, test, model_fn):
    X_train, y_train = train[NUMERIC_FEATURES + [SPECIES_COL]], train[TARGET].astype(int)
    X_test = test[NUMERIC_FEATURES + [SPECIES_COL]]
    model = model_fn()
    maybe_set_xgb_weight(model, y_train)
    pipe = Pipeline([("prep", make_multi_preprocessor()), ("clf", model)])
    pipe.fit(X_train, y_train)
    p_test_all, p_train_all = pipe.predict_proba(X_test)[:, 1], pipe.predict_proba(X_train)[:, 1]

    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask_test = (test[SPECIES_COL] == species).values
        mask_train = (train[SPECIES_COL] == species).values
        y_train_sp = train.loc[mask_train, TARGET].astype(int)
        if y_train_sp.empty:
            continue
        m = full_metrics(test_sp[TARGET].astype(int), p_test_all[mask_test], y_train_sp, p_train_all[mask_train])
        m["species"], m["approach"] = species, "multi"
        rows.append(m)
    return pd.DataFrame(rows)


def plot_diff_heatmap(pivot_diff, title, path, cbar_label):
    fig, ax = plt.subplots(figsize=(10, 5))
    vmax = np.nanmax(np.abs(pivot_diff.values)) if pivot_diff.size else 1
    im = ax.imshow(pivot_diff.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot_diff.columns)))
    ax.set_xticklabels(pivot_diff.columns)
    ax.set_yticks(range(len(pivot_diff.index)))
    ax.set_yticklabels(pivot_diff.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=cbar_label)
    for i in range(pivot_diff.shape[0]):
        for j in range(pivot_diff.shape[1]):
            val = pivot_diff.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="black", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def section_full_metrics(train, test, output_dir):
    print("\n" + "=" * 70)
    print("SECTION 5 — FULL METRICS (Brier / F1 / sensitivity / specificity) + HEATMAPS")
    print("=" * 70)
    models = build_models()
    all_rows = []
    for model_name, model_fn in models.items():
        print(f"Running {model_name} ...")
        single_df = run_single_full(train, test, model_fn)
        single_df["model"] = model_name
        multi_df = run_multi_full(train, test, model_fn)
        multi_df["model"] = model_name
        all_rows += [single_df, multi_df]

    full = pd.concat(all_rows, ignore_index=True)
    full.to_csv(os.path.join(output_dir, "full_metrics_per_species.csv"), index=False)

    for metric_col, label, fname in [
        ("log_loss", "log-loss diff (single - multi)", "heatmap_logloss_diff.png"),
        ("f1_at_opt", "F1 diff (single - multi) at train-optimal threshold", "heatmap_f1_diff.png"),
    ]:
        wide = full.pivot_table(index="species", columns=["model", "approach"], values=metric_col)
        diff_cols = {m: wide[(m, "single")] - wide[(m, "multi")]
                     for m in models if (m, "single") in wide.columns and (m, "multi") in wide.columns}
        plot_diff_heatmap(pd.DataFrame(diff_cols), f"{label}\n(blue = multi wins, red = single wins)",
                           os.path.join(output_dir, fname), label)

    overall_rows = []
    for model_name in models:
        for approach in ["single", "multi"]:
            df = full[(full.model == model_name) & (full.approach == approach)]
            overall_rows.append({
                "model": model_name, "approach": approach,
                "mean_log_loss": df["log_loss"].mean(), "mean_auc": df["auc"].mean(),
                "mean_brier": df["brier"].mean(), "mean_f1_at_opt": df["f1_at_opt"].mean(),
                "mean_sensitivity_at_opt": df["sensitivity_at_opt"].mean(),
                "mean_specificity_at_opt": df["specificity_at_opt"].mean(),
            })
    summary_df = pd.DataFrame(overall_rows)
    summary_df.to_csv(os.path.join(output_dir, "metrics_summary.csv"), index=False)
    print("\n----- Overall metrics summary -----")
    print(summary_df.to_string(index=False))


# ========================================================================
# MAIN
# ========================================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    print(f"Train: {train.shape}, Test: {test.shape}")
    print(f"Models included: {model_names()}")

    if RUN_BASELINE:
        section_baseline(train, test, OUTPUT_DIR)
    if RUN_REPEATED_SPATIAL_CV:
        section_repeated_spatial_cv(train, test, OUTPUT_DIR)
    if RUN_HYPERPARAM_TUNING:
        section_hyperparameter_tuning(train, test, OUTPUT_DIR)
    if RUN_FEATURE_IMPORTANCE:
        section_feature_importance(train, test, OUTPUT_DIR)
    if RUN_FULL_METRICS:
        section_full_metrics(train, test, OUTPUT_DIR)

    print("\nAll requested sections complete. Outputs saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

Train: (4102, 14), Test: (1026, 14)
Models included: ['LogisticRegression', 'RandomForest', 'MLP', 'XGBoost', 'GAM']

SECTION 1 — BASELINE: single vs multi vs multi+interactions
Running LogisticRegression ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/All model/outputs/LogisticRegression_comparison.png
  single: 0.4456 | multi: 0.5190 | multi+int: 0.4531
Running RandomForest ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/All model/outputs/RandomForest_comparison.png
  single: 0.1720 | multi: 0.1765 | multi+int: 0.1737
Running MLP ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/All model/outputs/MLP_comparison.png
  single: 0.3757 | multi: 0.2669 | multi+int: 0.2173
Running XGBoost ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/All model/outputs/XGBoost_comparison.png
  single: 0.2006 | multi: 0.2944 | multi+int: 0.2669
Running GAM ...
[saved] /U